# GPT 프롬프트 캐싱(KV 캐시) 활용 예제

OpenAI API의 **프롬프트 캐싱(Prompt Caching)** 은 서버 측 KV 캐시를 활용해,
반복되는 긴 프롬프트 앞부분(프리픽스)의 연산을 재사용하는 기능입니다.

핵심 규칙 (공식 문서 기준):
- **1024 토큰 이상**인 프롬프트부터 자동 적용 (별도 옵션 없이 동작, 128 토큰 단위로 캐시)
- **프롬프트 앞부분이 완전히 동일**해야 캐시 적중 → 고정 내용(지시문, 문서)은 앞에, 변하는 내용(사용자 질문)은 뒤에
- `prompt_cache_key` 파라미터로 같은 프리픽스의 요청을 같은 캐시로 라우팅하면 적중률 향상
- 캐시 적중량은 응답의 `usage.input_tokens_details.cached_tokens`로 확인
- 캐시된 입력 토큰은 **할인된 요금**으로 과금되고 응답 지연도 줄어듦
- 캐시 유지 시간: 기본 **24시간** (2026년 5월부터, ZDR 미사용 조직 기준 — `prompt_cache_retention: "24h"`로 명시 가능. 과거 인메모리 기본값은 미사용 5~10분, 최대 1시간이었음)

In [1]:
import time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
MODEL = "gpt-5-nano"

## 1. 긴 고정 프리픽스 만들기 (1024 토큰 이상)

캐싱이 발동하려면 프롬프트가 1024 토큰을 넘어야 합니다.
고객지원 상담원 정책 문서를 고정 지시문(instructions)으로 사용한다고 가정합니다.

In [2]:
CATEGORIES = ["배송 지연", "제품 불량", "환불 요청", "교환 요청", "결제 오류",
              "계정 문제", "쿠폰/할인", "멤버십", "A/S 접수", "기타 문의"]

rules = []
for i, cat in enumerate(CATEGORIES, start=1):
    rules.append(
        f"규정 {i} — [{cat}] 문의 처리 지침: "
        f"{cat} 유형의 문의가 접수되면 먼저 주문번호와 고객 본인 여부를 확인한다. "
        f"확인이 완료되면 {cat} 전용 처리 절차에 따라 1차 안내를 제공하고, "
        f"고객이 불만족할 경우 상급 상담원에게 에스컬레이션한다. "
        f"환불 또는 보상이 필요한 경우 결제 수단과 동일한 방법으로 영업일 기준 3~5일 내 처리하며, "
        f"5만 원을 초과하는 보상은 팀장 승인을 반드시 받아야 한다. "
        f"모든 상담 내역은 CRM에 기록하고, 고객에게는 처리 결과를 문자와 이메일로 이중 안내한다. "
        f"응대 시에는 항상 존댓말을 사용하고, 회사 내부 정보나 다른 고객의 정보는 절대 언급하지 않는다."
    )

LONG_INSTRUCTIONS = (
    "당신은 진주ICT 온라인 쇼핑몰의 고객지원 상담원입니다. "
    "아래의 사내 고객지원 정책을 반드시 준수하여 답변하세요. "
    "정책에 없는 내용은 임의로 약속하지 말고 확인 후 안내하겠다고 답하세요.\n\n"
    + "\n\n".join(rules)
)

print(f"지시문 길이: {len(LONG_INSTRUCTIONS):,}자 (대략 1024 토큰 이상)")

지시문 길이: 3,462자 (대략 1024 토큰 이상)


## 2. 사용량 확인용 헬퍼 함수

매 호출 후 `usage`에서 전체 입력 토큰과 캐시에서 읽은 토큰(`cached_tokens`)을 출력합니다.

In [5]:
def ask(question: str, label: str) -> None:
    start = time.perf_counter()
    response = client.responses.create(
        model=MODEL,
        instructions=LONG_INSTRUCTIONS,      # 고정 프리픽스 (캐시 대상)
        input=question,                      # 변하는 부분은 항상 뒤에
        prompt_cache_key="jinju-support-demo",  # 같은 프리픽스 요청을 같은 캐시로 라우팅
    )
    elapsed = time.perf_counter() - start

    usage = response.usage
    details = usage.input_tokens_details
    cached = getattr(details, "cached_tokens", 0) or 0
    hit_rate = cached / usage.input_tokens * 100 if usage.input_tokens else 0

    print(f"━━━ {label} ━━━")
    print(f"  질문: {question}")
    print(f"  입력 토큰: {usage.input_tokens:,} | 캐시 적중: {cached:,} ({hit_rate:.0f}%) | 출력 토큰: {usage.output_tokens:,}")
    print(f"  응답 시간: {elapsed:.2f}초")
    print(f"  답변(앞부분): {response.output_text[:80]}...\n")

## 3. 첫 호출 — 캐시 기록 (cached_tokens = 0)

첫 요청은 캐시가 비어 있으므로 `cached_tokens`가 0입니다. 이때 서버가 프리픽스를 캐시에 기록합니다.

In [6]:
ask("배송이 일주일째 안 오고 있어요. 어떻게 해야 하나요?", "1차 호출 (캐시 기록)")

━━━ 1차 호출 (캐시 기록) ━━━
  질문: 배송이 일주일째 안 오고 있어요. 어떻게 해야 하나요?
  입력 토큰: 1,912 | 캐시 적중: 1,792 (94%) | 출력 토큰: 229
  응답 시간: 5.54초
  답변(앞부분): 불편을 드려 죄송합니다. 배송 지연 문의는 먼저 **주문번호와 고객님 본인 여부 확인**이 필요합니다.

아래 정보를 알려주시면 확인 후 배송 ...



## 4. 두 번째 호출부터 — 캐시 적중 (cached_tokens > 0)

같은 지시문(프리픽스)에 **질문만 바꿔서** 다시 호출하면,
프리픽스 부분이 캐시에서 재사용되어 `cached_tokens`가 크게 잡히고 응답도 빨라집니다.

In [7]:
time.sleep(2)  # 캐시 전파를 위한 짧은 대기

ask("받은 제품이 불량인데 교환하고 싶어요.", "2차 호출 (캐시 적중 기대)")
ask("7만 원짜리 상품 환불은 얼마나 걸리나요?", "3차 호출 (캐시 적중 기대)")
ask("멤버십 등급은 어떻게 올라가나요?", "4차 호출 (캐시 적중 기대)")

━━━ 2차 호출 (캐시 적중 기대) ━━━
  질문: 받은 제품이 불량인데 교환하고 싶어요.
  입력 토큰: 1,908 | 캐시 적중: 1,792 (94%) | 출력 토큰: 208
  응답 시간: 5.65초
  답변(앞부분): 불편을 드려 죄송합니다. 제품 불량으로 교환 접수를 도와드리기 위해 먼저 확인이 필요합니다.

1. **주문번호**를 알려주시겠어요?  
2. ...

━━━ 3차 호출 (캐시 적중 기대) ━━━
  질문: 7만 원짜리 상품 환불은 얼마나 걸리나요?
  입력 토큰: 1,909 | 캐시 적중: 1,792 (94%) | 출력 토큰: 276
  응답 시간: 5.68초
  답변(앞부분): 고객님, 7만 원 상품의 환불은 **주문번호와 고객님 본인 여부 확인 후**, 환불 접수가 완료되면 **결제하신 수단과 동일한 방법으로 영업일 ...

━━━ 4차 호출 (캐시 적중 기대) ━━━
  질문: 멤버십 등급은 어떻게 올라가나요?
  입력 토큰: 1,907 | 캐시 적중: 1,792 (94%) | 출력 토큰: 228
  응답 시간: 6.11초
  답변(앞부분): 안녕하세요, 진주ICT 고객지원입니다.

멤버십 등급 관련 문의는 먼저 **주문번호와 고객님 본인 여부 확인**이 필요합니다.  
아래 정보를 ...



> 💡 `cached_tokens`가 0으로 나오면 몇 초 뒤 위 셀을 다시 실행해보세요.
> 캐시 기록이 전파되기 전에 후속 요청이 도착하면 적중하지 않을 수 있습니다.

## 5. 캐시가 깨지는 경우 — 프리픽스가 달라지면 무효

캐시는 **프롬프트 앞부분이 완전히 동일**할 때만 적중합니다.
지시문 맨 앞에 한 글자만 추가해도 프리픽스가 달라져서 캐시를 전혀 못 씁니다.

In [6]:
start = time.perf_counter()
response = client.responses.create(
    model=MODEL,
    instructions="※ " + LONG_INSTRUCTIONS,  # 맨 앞에 두 글자 추가 → 프리픽스 불일치
    input="배송 조회는 어디서 하나요?",
    prompt_cache_key="jinju-support-demo",
)
elapsed = time.perf_counter() - start

cached = getattr(response.usage.input_tokens_details, "cached_tokens", 0) or 0
print(f"프리픽스를 바꾼 호출 → 캐시 적중: {cached:,} 토큰 (응답 {elapsed:.2f}초)")
print("앞부분이 1024 토큰 이상 동일하지 않으면 캐시 적중량이 0이 되거나 크게 줄어듭니다.")

프리픽스를 바꾼 호출 → 캐시 적중: 0 토큰 (응답 7.70초)
앞부분이 1024 토큰 이상 동일하지 않으면 캐시 적중량이 0이 되거나 크게 줄어듭니다.


## 정리 — 캐싱 베스트 프랙티스

1. **고정 내용은 앞에, 변하는 내용은 뒤에** — 시스템 지시문·참고 문서·도구 정의를 프롬프트 앞쪽에 고정 배치
2. **`prompt_cache_key`를 일관되게** — 같은 프리픽스를 쓰는 요청 그룹마다 동일한 키 사용 (키당 분당 15요청 이하 권장)
3. **1024 토큰 미만이면 캐싱 미적용** — 짧은 프롬프트는 캐싱 대상이 아님
4. **`cached_tokens` 모니터링** — 운영 시 캐시 적중률을 지표로 관리 (최신 모델은 `cache_write_tokens`도 제공)
5. **유지 시간 확인** — 현재 기본 보존은 24시간(확장 보존). 구형 인메모리 캐시는 미사용 5~10분이면 만료됐으므로, 문서의 `prompt_cache_retention` 설정을 확인할 것